
# Healthcare Patient Analytics

## Overview
This notebook implements patient and encounter analytics
using PySpark, Spark SQL, and Delta Lake on Databricks.

## Data Sources
Synthetic patient and healthcare encounter datasets.
No real patient information is used.

## Data Processing
- Explicit schema definitions
- Primary key validation
- Data quality checks
- Referential integrity validation
- PySpark joins and aggregations
- Delta Lake managed tables
- SQL-based healthcare analytics

## Data Outputs
- workspace.healthcare_analytics.patients
- workspace.healthcare_analytics.encounters
- workspace.healthcare_analytics.patient_analytics

## Quality Assurance
The notebook validates record counts, uniqueness,
referential integrity, financial reconciliation,
and Delta table storage formats.

## Technology Stack
- Databricks
- Apache Spark
- PySpark
- Spark SQL
- Delta Lake
- Python


In [0]:
%python

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

from datetime import date
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("healthcare_analytics")

print("Databricks healthcare analytics initialized")
print("Spark version:", spark.version)
print("Current catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current schema:", spark.sql("SELECT current_schema()").first()[0])


Databricks healthcare analytics initialized
Spark version: 4.2.0
Current catalog: workspace
Current schema: default


In [0]:
%python

catalog_name = "workspace"
schema_name = "healthcare_analytics"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`"
)

spark.sql(
    f"USE CATALOG `{catalog_name}`"
)

spark.sql(
    f"USE SCHEMA `{schema_name}`"
)

print("Healthcare analytics schema initialized")
print("Active catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Active schema:", spark.sql("SELECT current_schema()").first()[0])

spark.sql("SHOW SCHEMAS").show(truncate=False)


Healthcare analytics schema initialized
Active catalog: workspace
Active schema: healthcare_analytics
+--------------------+
|databaseName        |
+--------------------+
|default             |
|healthcare_analytics|
|information_schema  |
+--------------------+



In [0]:
%python

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType
)

from datetime import date

# Define the synthetic patient data schema
patient_schema = StructType([
    StructField("patient_id", StringType(), False),
    StructField("patient_name", StringType(), False),
    StructField("age", IntegerType(), False),
    StructField("gender", StringType(), False),
    StructField("city", StringType(), False),
    StructField("registration_date", DateType(), False)
])

# Generate synthetic healthcare patient records
patient_data = [
    ("P001", "Patient Alpha", 34, "Female", "Bengaluru", date(2026, 1, 10)),
    ("P002", "Patient Beta", 46, "Male", "Mysuru", date(2026, 1, 12)),
    ("P003", "Patient Gamma", 28, "Female", "Hyderabad", date(2026, 2, 3)),
    ("P004", "Patient Delta", 62, "Male", "Chennai", date(2026, 2, 15)),
    ("P005", "Patient Epsilon", 51, "Female", "Pune", date(2026, 3, 4))
]

# Create the PySpark DataFrame
patients_df = spark.createDataFrame(
    patient_data,
    schema=patient_schema
)

# Display the synthetic patient dataset
display(patients_df)

# Verify record count and schema
print("Total patient records:", patients_df.count())

patients_df.printSchema()


patient_id,patient_name,age,gender,city,registration_date
P001,Patient Alpha,34,Female,Bengaluru,2026-01-10
P002,Patient Beta,46,Male,Mysuru,2026-01-12
P003,Patient Gamma,28,Female,Hyderabad,2026-02-03
P004,Patient Delta,62,Male,Chennai,2026-02-15
P005,Patient Epsilon,51,Female,Pune,2026-03-04


Total patient records: 5
root
 |-- patient_id: string (nullable = false)
 |-- patient_name: string (nullable = false)
 |-- age: integer (nullable = false)
 |-- gender: string (nullable = false)
 |-- city: string (nullable = false)
 |-- registration_date: date (nullable = false)



In [0]:
%python

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    DateType
)

from datetime import date

# Define the healthcare encounter schema
encounter_schema = StructType([
    StructField("encounter_id", StringType(), False),
    StructField("patient_id", StringType(), False),
    StructField("department", StringType(), False),
    StructField("encounter_date", DateType(), False),
    StructField("treatment_cost", DoubleType(), False)
])

# Generate synthetic healthcare encounter records
encounter_data = [
    ("E001", "P001", "Cardiology", date(2026, 4, 1), 2500.0),
    ("E002", "P002", "Neurology", date(2026, 4, 2), 4200.0),
    ("E003", "P001", "General Medicine", date(2026, 4, 5), 1200.0),
    ("E004", "P003", "Orthopedics", date(2026, 4, 6), 3600.0),
    ("E005", "P004", "Cardiology", date(2026, 4, 7), 5400.0),
    ("E006", "P005", "General Medicine", date(2026, 4, 8), 1800.0),
    ("E007", "P002", "Cardiology", date(2026, 4, 9), 3100.0)
]

# Create the encounter DataFrame
encounters_df = spark.createDataFrame(
    encounter_data,
    schema=encounter_schema
)

# Display the encounter dataset
display(encounters_df)

# Verify the number of records and schema
print("Total encounter records:", encounters_df.count())

encounters_df.printSchema()


encounter_id,patient_id,department,encounter_date,treatment_cost
E001,P001,Cardiology,2026-04-01,2500.0
E002,P002,Neurology,2026-04-02,4200.0
E003,P001,General Medicine,2026-04-05,1200.0
E004,P003,Orthopedics,2026-04-06,3600.0
E005,P004,Cardiology,2026-04-07,5400.0
E006,P005,General Medicine,2026-04-08,1800.0
E007,P002,Cardiology,2026-04-09,3100.0


Total encounter records: 7
root
 |-- encounter_id: string (nullable = false)
 |-- patient_id: string (nullable = false)
 |-- department: string (nullable = false)
 |-- encounter_date: date (nullable = false)
 |-- treatment_cost: double (nullable = false)



In [0]:
%python

from pyspark.sql import functions as F
import logging

logger = logging.getLogger("healthcare_analytics")

# Reusable data quality validation function
def validate_dataset(df, primary_key):
    total_records = df.count()

    null_keys = df.filter(
        F.col(primary_key).isNull()
    ).count()

    duplicate_keys = (
        df.groupBy(primary_key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if total_records == 0:
        raise ValueError("Dataset cannot be empty")

    if null_keys > 0:
        raise ValueError(
            f"Null primary keys found: {null_keys}"
        )

    if duplicate_keys > 0:
        raise ValueError(
            f"Duplicate primary keys found: {duplicate_keys}"
        )

    logger.info(
        "Validated %s records using primary key %s",
        total_records,
        primary_key
    )

    return total_records


# Validate patient records
patient_count = validate_dataset(
    patients_df,
    "patient_id"
)

assert patient_count == 5, "Unexpected patient count"

# Validate encounter records
encounter_count = validate_dataset(
    encounters_df,
    "encounter_id"
)

assert encounter_count == 7, "Unexpected encounter count"

# Validate patient ages
invalid_ages = patients_df.filter(
    (F.col("age") < 0) |
    (F.col("age") > 120) |
    F.col("age").isNull()
).count()

assert invalid_ages == 0, "Invalid patient ages found"

# Validate treatment costs
invalid_costs = encounters_df.filter(
    (F.col("treatment_cost") < 0) |
    F.col("treatment_cost").isNull() |
    F.isnan("treatment_cost")
).count()

assert invalid_costs == 0, "Invalid treatment costs found"

# Validate patient-to-encounter relationships
orphan_encounters = encounters_df.join(
    patients_df.select("patient_id"),
    on="patient_id",
    how="left_anti"
)

orphan_count = orphan_encounters.count()

assert orphan_count == 0, "Orphan encounters found"

print("PASS: Patient primary key validation")
print("PASS: Encounter primary key validation")
print("PASS: Patient age validation")
print("PASS: Treatment cost validation")
print("PASS: Referential integrity validation")
print("All healthcare data quality checks passed")


INFO:healthcare_analytics:Validated 5 records using primary key patient_id
INFO:healthcare_analytics:Validated 7 records using primary key encounter_id


PASS: Patient primary key validation
PASS: Encounter primary key validation
PASS: Patient age validation
PASS: Treatment cost validation
PASS: Referential integrity validation
All healthcare data quality checks passed


In [0]:
%python

# Define fully qualified Delta table names
catalog_name = "workspace"
schema_name = "healthcare_analytics"

patients_table = f"{catalog_name}.{schema_name}.patients"
encounters_table = f"{catalog_name}.{schema_name}.encounters"

# Save validated patient records as a managed Delta table
patients_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(patients_table)

print(f"Patient Delta table created: {patients_table}")

# Save validated encounter records as a managed Delta table
encounters_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(encounters_table)

print(f"Encounter Delta table created: {encounters_table}")

# Verify the persisted record counts
persisted_patients = spark.table(patients_table).count()
persisted_encounters = spark.table(encounters_table).count()

assert persisted_patients == 5, "Patient table count mismatch"
assert persisted_encounters == 7, "Encounter table count mismatch"

print("Persisted patient records:", persisted_patients)
print("Persisted encounter records:", persisted_encounters)
print("PASS: Both Delta tables saved and verified")


Patient Delta table created: workspace.healthcare_analytics.patients
Encounter Delta table created: workspace.healthcare_analytics.encounters
Persisted patient records: 5
Persisted encounter records: 7
PASS: Both Delta tables saved and verified


In [0]:
%python

# Inspect the Delta table storage formats
for table_name in [
    "workspace.healthcare_analytics.patients",
    "workspace.healthcare_analytics.encounters"
]:
    table_details = spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    )

    table_format = table_details.select("format").first()[0]

    assert table_format == "delta", (
        f"Unexpected table format for {table_name}: {table_format}"
    )

    print(f"PASS: {table_name} uses Delta format")


PASS: workspace.healthcare_analytics.patients uses Delta format
PASS: workspace.healthcare_analytics.encounters uses Delta format


In [0]:
%python

from pyspark.sql import functions as F

# Read source data from persisted Delta tables
patients_source_df = spark.table(
    "workspace.healthcare_analytics.patients"
)

encounters_source_df = spark.table(
    "workspace.healthcare_analytics.encounters"
)

# Aggregate healthcare encounters by patient
encounter_summary_df = (
    encounters_source_df
    .groupBy("patient_id")
    .agg(
        F.count("encounter_id").alias("total_encounters"),
        F.round(
            F.sum("treatment_cost"), 2
        ).alias("total_treatment_cost"),
        F.max("encounter_date").alias("last_encounter_date")
    )
)

# Join patient details with encounter metrics
patient_analytics_df = (
    patients_source_df
    .join(
        encounter_summary_df,
        on="patient_id",
        how="left"
    )
    .select(
        "patient_id",
        "patient_name",
        "age",
        "gender",
        "city",
        F.coalesce(
            F.col("total_encounters"),
            F.lit(0)
        ).alias("total_encounters"),
        F.coalesce(
            F.col("total_treatment_cost"),
            F.lit(0.0)
        ).alias("total_treatment_cost"),
        "last_encounter_date"
    )
)

# Display the analytics dataset
display(
    patient_analytics_df.orderBy("patient_id")
)

print(
    "Total analytics records:",
    patient_analytics_df.count()
)


patient_id,patient_name,age,gender,city,total_encounters,total_treatment_cost,last_encounter_date
P001,Patient Alpha,34,Female,Bengaluru,2,3700.0,2026-04-05
P002,Patient Beta,46,Male,Mysuru,2,7300.0,2026-04-09
P003,Patient Gamma,28,Female,Hyderabad,1,3600.0,2026-04-06
P004,Patient Delta,62,Male,Chennai,1,5400.0,2026-04-07
P005,Patient Epsilon,51,Female,Pune,1,1800.0,2026-04-08


Total analytics records: 5


In [0]:
%python

# Define the target analytics table
analytics_table = (
    "workspace.healthcare_analytics.patient_analytics"
)

# Persist analytics results as a managed Delta table
patient_analytics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(analytics_table)

# Read the persisted analytics table
saved_analytics_df = spark.table(analytics_table)

# Verify record counts and financial metrics
analytics_count = saved_analytics_df.count()

total_encounters = saved_analytics_df.agg(
    F.sum("total_encounters")
).first()[0]

total_cost = saved_analytics_df.agg(
    F.sum("total_treatment_cost")
).first()[0]

assert analytics_count == 5, "Analytics record count mismatch"
assert total_encounters == 7, "Encounter count mismatch"
assert round(total_cost, 2) == 21800.00, (
    "Treatment cost reconciliation failed"
)

# Verify that the target table uses Delta format
table_format = spark.sql(
    f"DESCRIBE DETAIL {analytics_table}"
).select("format").first()[0]

assert table_format == "delta", "Target table is not Delta"

print("PASS: Patient analytics Delta table created")
print("Analytics records:", analytics_count)
print("Total encounters:", total_encounters)
print("Total treatment cost:", round(total_cost, 2))
print("Storage format:", table_format)


PASS: Patient analytics Delta table created
Analytics records: 5
Total encounters: 7
Total treatment cost: 21800.0
Storage format: delta


In [0]:
%python

catalog_name = "workspace"
schema_name = "healthcare_analytics"
table_name = "patient_analytics"

print("Checking healthcare analytics schema...")

spark.sql("""
    SHOW TABLES IN workspace.healthcare_analytics
""").show(truncate=False)

table_exists = spark.catalog.tableExists(
    f"{catalog_name}.{schema_name}.{table_name}"
)

print("Patient analytics table exists:", table_exists)

if table_exists:
    print("Patient analytics record count:",
          spark.table(
              "workspace.healthcare_analytics.patient_analytics"
          ).count())
else:
    print("Patient analytics table has not been created.")


Checking healthcare analytics schema...
+--------------------+-----------------+-----------+
|database            |tableName        |isTemporary|
+--------------------+-----------------+-----------+
|healthcare_analytics|encounters       |false      |
|healthcare_analytics|patient_analytics|false      |
|healthcare_analytics|patients         |false      |
+--------------------+-----------------+-----------+

Patient analytics table exists: True
Patient analytics record count: 5


In [0]:
%sql

SELECT
    city,
    COUNT(*) AS total_patients,
    SUM(total_encounters) AS total_encounters,
    ROUND(SUM(total_treatment_cost), 2) AS total_treatment_cost
FROM workspace.healthcare_analytics.patient_analytics
GROUP BY city
ORDER BY total_treatment_cost DESC;


city,total_patients,total_encounters,total_treatment_cost
Mysuru,1,2,7300.0
Chennai,1,1,5400.0
Bengaluru,1,2,3700.0
Hyderabad,1,1,3600.0
Pune,1,1,1800.0


Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT
    department,
    COUNT(*) AS total_encounters,
    COUNT(DISTINCT patient_id) AS unique_patients,
    ROUND(SUM(treatment_cost), 2) AS total_treatment_cost,
    ROUND(AVG(treatment_cost), 2) AS average_treatment_cost
FROM workspace.healthcare_analytics.encounters
GROUP BY department
ORDER BY total_treatment_cost DESC;


department,total_encounters,unique_patients,total_treatment_cost,average_treatment_cost
Cardiology,3,3,11000.0,3666.67
Neurology,1,1,4200.0,4200.0
Orthopedics,1,1,3600.0,3600.0
General Medicine,2,2,3000.0,1500.0


In [0]:
%sql

SELECT
    patient_id,
    patient_name,
    city,
    total_encounters,
    total_treatment_cost,
    last_encounter_date
FROM workspace.healthcare_analytics.patient_analytics
ORDER BY total_treatment_cost DESC;


patient_id,patient_name,city,total_encounters,total_treatment_cost,last_encounter_date
P002,Patient Beta,Mysuru,2,7300.0,2026-04-09
P004,Patient Delta,Chennai,1,5400.0,2026-04-07
P001,Patient Alpha,Bengaluru,2,3700.0,2026-04-05
P003,Patient Gamma,Hyderabad,1,3600.0,2026-04-06
P005,Patient Epsilon,Pune,1,1800.0,2026-04-08


In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM workspace.healthcare_analytics.patients)
        AS total_patients,

    (SELECT COUNT(*)
     FROM workspace.healthcare_analytics.encounters)
        AS total_encounters,

    (SELECT COUNT(*)
     FROM workspace.healthcare_analytics.patient_analytics)
        AS analytics_records,

    (SELECT ROUND(SUM(treatment_cost), 2)
     FROM workspace.healthcare_analytics.encounters)
        AS total_treatment_cost;


total_patients,total_encounters,analytics_records,total_treatment_cost
5,7,5,21800.0


In [0]:

tables = [
    "workspace.healthcare_analytics.patients",
    "workspace.healthcare_analytics.encounters",
    "workspace.healthcare_analytics.patient_analytics"
]

for table_name in tables:
    print(f"\nDelta transaction history: {table_name}")

    history_df = spark.sql(
        f"DESCRIBE HISTORY {table_name}"
    )

    display(
        history_df.select(
            "version",
            "timestamp",
            "operation"
        ).orderBy("version", ascending=False)
    )



Delta transaction history: workspace.healthcare_analytics.patients


version,timestamp,operation
0,2026-11-17T17:52:46.000Z,CREATE OR REPLACE TABLE AS SELECT



Delta transaction history: workspace.healthcare_analytics.encounters


version,timestamp,operation
0,2026-11-17T17:52:50.000Z,CREATE OR REPLACE TABLE AS SELECT



Delta transaction history: workspace.healthcare_analytics.patient_analytics


version,timestamp,operation
1,2026-11-17T17:56:25.000Z,CREATE OR REPLACE TABLE AS SELECT
0,2026-11-17T17:55:08.000Z,CREATE OR REPLACE TABLE AS SELECT


In [0]:

for table_name in tables:
    details = spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).first().asDict()

    assert details["format"] == "delta", (
        f"Invalid storage format: {table_name}"
    )

    print(f"PASS: {table_name}")
    print("Format:", details["format"])
    print("Files:", details["numFiles"])
    print("Size in bytes:", details["sizeInBytes"])
    print("-" * 50)


PASS: workspace.healthcare_analytics.patients
Format: delta
Files: 1
Size in bytes: 2113
--------------------------------------------------
PASS: workspace.healthcare_analytics.encounters
Format: delta
Files: 1
Size in bytes: 1962
--------------------------------------------------
PASS: workspace.healthcare_analytics.patient_analytics
Format: delta
Files: 1
Size in bytes: 2701
--------------------------------------------------


In [0]:

from pyspark.sql import functions as F

catalog = "workspace"
schema = "healthcare_analytics"

patients = spark.table(f"{catalog}.{schema}.patients")
encounters = spark.table(f"{catalog}.{schema}.encounters")
analytics = spark.table(f"{catalog}.{schema}.patient_analytics")

# Record count validation
assert patients.count() == 5
assert encounters.count() == 7
assert analytics.count() == 5

# Primary key uniqueness
assert patients.select("patient_id").distinct().count() == 5
assert encounters.select("encounter_id").distinct().count() == 7
assert analytics.select("patient_id").distinct().count() == 5

# Referential integrity
orphan_count = encounters.join(
    patients.select("patient_id"),
    on="patient_id",
    how="left_anti"
).count()

assert orphan_count == 0

# Financial reconciliation
source_cost = encounters.agg(
    F.sum("treatment_cost")
).first()[0]

analytics_cost = analytics.agg(
    F.sum("total_treatment_cost")
).first()[0]

assert round(source_cost, 2) == 21800.00
assert round(analytics_cost, 2) == 21800.00

# Encounter reconciliation
analytics_visits = analytics.agg(
    F.sum("total_encounters")
).first()[0]

assert analytics_visits == 7

# Verify all tables are Delta
for table_name in [
    f"{catalog}.{schema}.patients",
    f"{catalog}.{schema}.encounters",
    f"{catalog}.{schema}.patient_analytics"
]:
    storage_format = spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).select("format").first()[0]

    assert storage_format == "delta"

print("PASS: Patient record count")
print("PASS: Encounter record count")
print("PASS: Analytics record count")
print("PASS: Primary key uniqueness")
print("PASS: Referential integrity")
print("PASS: Financial reconciliation")
print("PASS: Encounter reconciliation")
print("PASS: Delta storage verification")
print("ALL HEALTHCARE ANALYTICS VALIDATIONS PASSED")


PASS: Patient record count
PASS: Encounter record count
PASS: Analytics record count
PASS: Primary key uniqueness
PASS: Referential integrity
PASS: Financial reconciliation
PASS: Encounter reconciliation
PASS: Delta storage verification
ALL HEALTHCARE ANALYTICS VALIDATIONS PASSED
